In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/tiwarikanaksuraj/train-csv/train (1).csv


In [2]:
df=pd.read_csv("/kaggle/input/datasets/tiwarikanaksuraj/train-csv/train (1).csv")


# milestone-1 
#### Q1 Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?  

In [3]:
x=df['answer'].value_counts().max() +  df['answer'].value_counts().min()

print("Sum of Most frequent and least frequent options in answer column =",x)

Sum of Most frequent and least frequent options in answer column = 814


### Q2 After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?   

In [4]:
import string
has_punctuation = df["prompt"].apply(
    lambda text: any(c in string.punctuation for c in text)
)
print(has_punctuation.unique())

[ True]


In [5]:
import string
def clean_prompt(text):
    text=text.lower()
    translator=str.maketrans('','',string.punctuation) #nothing,nothing,delete punctuation 
    text=text.translate(translator)
    return text
    

In [6]:
df["prompt"]=df["prompt"].apply(clean_prompt)
#verifying 
has_punctuation = df["prompt"].apply(
    lambda text: any(c in string.punctuation for c in text)
)
print(has_punctuation.unique())

[False]


In [7]:
vocab=set()
for text in df["prompt"]:
    vocab.update(text.split())

print(len(vocab))

859


### Q3 Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  


In [8]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import string

temp=df[df["id"]==1]["prompt"].iloc[0].split()
filtered_words=[i for i in temp if i not in ENGLISH_STOP_WORDS] 
print("unfiltered string :",temp)
print("filtered string list : ",filtered_words)
print("length of list ",len(filtered_words))

unfiltered string : ['pick', 'the', 'best', 'possible', 'answer', 'what', 'is', 'martin', 'heideggers', 'view', 'on', 'the', 'relationship', 'between', 'time', 'and', 'human', 'existence', 'among', 'the', 'listed', 'options']
filtered string list :  ['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
length of list  13


### Q4 Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
combinelist=pd.concat(
    [df["prompt"],
    df["A"],
    df["B"],
     df["C"],
     df["D"],
     df["E"]
    ])
vectorizer=TfidfVectorizer(stop_words="english")
vectorizer.fit(combinelist)

print(len(vectorizer.vocabulary_))


2807


### Q5 Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

In [10]:
from sklearn.metrics.pairwise import cosine_similarity
row=df[df["id"]==1].iloc[0]
prompt=row["prompt"]
optA=row["A"]
prompt_vec=vectorizer.transform([prompt])
option_vec=vectorizer.transform([optA])
print(f"Similarity score : {cosine_similarity(prompt_vec,option_vec)[0,0]:.4f}")

Similarity score : 0.1682


### Q6 For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.  

In [11]:
def transform_columns(df, vectorizer):
    return (
        vectorizer.transform(df['prompt']),
        vectorizer.transform(df['A']),
        vectorizer.transform(df['B']),
        vectorizer.transform(df['C']),
        vectorizer.transform(df['D']),
        vectorizer.transform(df['E'])
    )

    
prompt_vec, A_vec, B_vec, C_vec, D_vec, E_vec = transform_columns(df, vectorizer)

prompt_vec.shape #where rows,no_of_unnique_words

(2000, 2807)

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

scores = {
    'A': cosine_similarity(prompt_vec, A_vec).diagonal(),
    'B': cosine_similarity(prompt_vec, B_vec).diagonal(),
    'C': cosine_similarity(prompt_vec, C_vec).diagonal(),
    'D': cosine_similarity(prompt_vec, D_vec).diagonal(),
    'E': cosine_similarity(prompt_vec, E_vec).diagonal()
}

correct_pred=0
for i in range(len(scores["A"])):
    row_scores={
        'A': scores['A'][i],
        'B': scores['B'][i],
        'C': scores['C'][i],
        'D': scores['D'][i],
        'E': scores['E'][i]  
    }
    best_opt='A'
    for option in ['B','C','D','E']:
        if row_scores[option] > row_scores[best_opt]:
            best_opt=option

    if (best_opt == df.loc[i,'answer']):
        correct_pred+=1

accuracy=(correct_pred/df.shape[0])*100
print(f"Accuracy of cosimilarity : {accuracy}")

Accuracy of cosimilarity : 12.5


### Q9 The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [13]:
def map3(y_true, top3_preds):
    score = 0

    for actual, pred in zip(y_true, top3_preds):
        if actual == pred[0]:
            score += 1.0

        elif actual == pred[1]:
            score += 0.5

        elif actual == pred[2]:
            score += 1/3

    return score / len(y_true)

In [14]:
counts = df['answer'].value_counts()

top3 = counts.index[:3].tolist()

print(top3)

['B', 'C', 'A']


In [15]:
y_pred = [top3] * len(df)

In [16]:
print("Majority class baseline : ",round(map3(df["answer"],y_pred),3))

Majority class baseline :  0.421


In [17]:
y_pred=[]
for i in range(len(scores["A"])):
    row_scores={
        'A': scores['A'][i],
        'B': scores['B'][i],
        'C': scores['C'][i],
        'D': scores['D'][i],
        'E': scores['E'][i]  
    }
    top3=[]
   
    for k in range(3):
        best_opt=None
        best_score=-1
        for option,score in row_scores.items():
            if score > best_score:
                best_score=score
                best_opt=option
        top3.append(best_opt)
        row_scores.pop(best_opt)
    
    y_pred.append(top3)    
print(len(y_pred))

2000


In [18]:
print(round(map3(df['answer'], y_pred),2))

0.29


# Milestone 2 
## Q1 Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.


In [19]:
from datasets import load_dataset 
dataset=load_dataset("csv",data_files="/kaggle/input/datasets/tiwarikanaksuraj/train-csv/train (1).csv")
train=dataset["train"]

train=train.map(
    lambda x : {
        "combined_text":x["prompt"]+""+x["A"]
    }
)

# Print first row

# Length of row 51
print(len(train[51]["combined_text"]))
print(type(list(train["prompt"])))


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

613
<class 'list'>


## Q2 Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?  

In [20]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print(tokenizer.vocab_size)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

30522


## Q3 Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.  

In [21]:
print("SEP token ",tokenizer.sep_token,tokenizer.sep_token_id)

SEP token  [SEP] 102


## Q4 Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors). What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [22]:
encodings=tokenizer(list(train["prompt"]),padding="max_length",truncation=True,max_length=128,return_tensors="pt")
print(encodings["input_ids"].shape)

torch.Size([2000, 128])


## Q5  A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer. In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?  

In [23]:
from transformers import BertModel
from transformers import logging
logging.set_verbosity_error()


model = BertModel.from_pretrained("bert-base-uncased")

print(model.config.hidden_size)
print(model.config.num_attention_heads)
print(model.config.hidden_size // model.config.num_attention_heads)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

768
12
64


## Q6 Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object. What is the exact shape of the last_hidden_state tensor returned? 

Note: We follow zero-indexing here

In [24]:
from transformers import  AutoModel
model = AutoModel.from_pretrained("bert-base-uncased")
text = train[0]["prompt"]
print(text)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.


In [25]:
inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

torch.Size([1, 31, 768])


  ## Q7 Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).  

In [26]:
cls_embedding = outputs.last_hidden_state[0, 0]
print(cls_embedding[:5] )

tensor([-0.4677, -0.0754, -0.2019, -0.0071, -0.4480], grad_fn=<SliceBackward0>)


In [27]:
answer = cls_embedding[:5].sum().item()
print(round(answer,4))

-1.2001



## Q8 Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0). What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places). 

In [28]:
from transformers import AutoModel
import torch

model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

text = "Light-ion fusion is a technique."

# Use your existing tokenizer
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

# Tokens (to find the index of "fusion")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(tokens)

# Last layer, first head
last_attention = outputs.attentions[-1]
head0 = last_attention[0, 0]

fusion_index = tokens.index("fusion")

weight = head0[0, fusion_index].item()

print("Attention weight:", round(weight, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Attention weight: 0.1025


## Q9 Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.

In [29]:
from sentence_transformers import SentenceTransformer, util

# Load model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Get row 0
prompt = train[0]["prompt"]
option_b = train[0]["B"]


# Generate embeddings
prompt_embedding = model.encode(prompt, convert_to_tensor=True)
option_b_embedding = model.encode(option_b, convert_to_tensor=True)

# Cosine similarity
similarity = util.cos_sim(prompt_embedding, option_b_embedding)

print(similarity)
print(round(similarity.item(), 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

tensor([[0.7658]], device='cuda:0')
0.7658


## Q10 Build two complete ranking pipelines evaluating every row in train.csv.

Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set? 

Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?  

In [30]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Load data


letters = ["A","B","C","D","E"]

def map3(actual, predicted):
    score = 0

    for a, p in zip(actual, predicted):
        if a == p[0]:
            score += 1
        elif a == p[1]:
            score += 1/2
        elif a == p[2]:
            score += 1/3

    return score / len(actual)


# TF-IDF PIPELINE


corpus = []

for row in train:
    corpus.append(row["prompt"])
    for l in letters:
        corpus.append(row[l])

vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(corpus)

tfidf_predictions = []

for row in train:

    prompt_vec = vectorizer.transform([row["prompt"]])

    scores = []

    for l in letters:
        option_vec = vectorizer.transform([row[l]])
        score = (prompt_vec @ option_vec.T).toarray()[0][0]
        scores.append(score)

    order = np.argsort(scores)[::-1]
    tfidf_predictions.append([letters[i] for i in order[:3]])

     

In [31]:

###################################################
# MiniLM PIPELINE
###################################################

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2",device="cuda")

minilm_predictions = []

for row in train:

    prompt_embedding = model.encode(
        row["prompt"],
        convert_to_tensor=True
    )

    scores = []

    for l in letters:

        option_embedding = model.encode(
            row[l],
            convert_to_tensor=True
        )

        score = util.cos_sim(
            prompt_embedding,
            option_embedding
        ).item()
        scores.append(score)

    order = np.argsort(scores)[::-1]
    minilm_predictions.append([letters[i] for i in order[:3]])



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [32]:

# RESULTS
answers = train["answer"]

map3_minilm = map3(answers, minilm_predictions)

count = 0

for ans, tfidf_top3, minilm_top3 in zip(
    answers,
    tfidf_predictions,
    minilm_predictions
):
    if ans not in tfidf_top3 and ans in minilm_top3:
        count += 1

print("MiniLM MAP@3 =", round(map3_minilm,6))
print("Count =", count)

MiniLM MAP@3 = 0.423083
Count = 575


In [33]:
from transformers import pipeline

# Initialize pipeline (defaults to facebook/bart-large-mnli)
classifier = pipeline("zero-shot-classification")

# Prompt from row index 1
sequence = train[1]["prompt"]

# Candidate labels = Options A, B, C
candidate_labels = [
    train[1]["A"],
    train[1]["B"],
    train[1]["C"]
]

# Classify
result = classifier(sequence, candidate_labels)

# View full result
print(result)

# Top-ranked probability
print(round(result["scores"][0], 4))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

In [34]:
from transformers import pipeline

# Initialize pipeline (defaults to facebook/bart-large-mnli)
classifier = pipeline("zero-shot-classification")

# Prompt from row index 1
sequence = train[1]["prompt"]

# Candidate labels = Options A, B, C
candidate_labels = [
    train[1]["A"],
    train[1]["B"],
    train[1]["C"]
]

# Classify
result = classifier(sequence, candidate_labels)

# View full result
print(result)

# Top-ranked probability
print(round(result["scores"][0], 4))

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

In [35]:

# Softmax (default)
result_softmax = classifier(
    sequence,
    candidate_labels
)

# Independent sigmoid probabilities
result_sigmoid = classifier(
    sequence,
    candidate_labels,
    multi_label=True
)

softmax_sum = sum(result_softmax["scores"])
sigmoid_sum = sum(result_sigmoid["scores"])

difference = abs(softmax_sum - sigmoid_sum)

print("Softmax scores:", result_softmax["scores"])
print("Sigmoid scores:", result_sigmoid["scores"])

print("Softmax sum:", softmax_sum)
print("Sigmoid sum:", sigmoid_sum)
print("Answer:", round(difference, 4))

Softmax scores: [0.4574522376060486, 0.2750644385814667, 0.26748329401016235]
Sigmoid scores: [0.00046926175127737224, 2.063553074549418e-05, 1.9700279153767042e-05]
Softmax sum: 0.9999999701976776
Sigmoid sum: 0.0005095975611766335
Answer: 0.9995


In [36]:
from transformers import pipeline

# Load FLAN-T5 Small
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small"
)

# Construct the prompt
text = (
    f"Question: {train[0]['prompt']}. "
    f"Is the correct answer A: {train[0]['A']} "
    f"or B: {train[0]['B']}? "
    f"Answer with just the letter A or B."
)

# Generate
result = generator(text, max_new_tokens=5)


print(result)

config.json: 0.00B [00:00, ?B/s]

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"

In [37]:
import transformers
print(transformers.__version__)

5.0.0


In [ ]:
!pip install -q "transformers==4.46.3"

In [38]:
import sys

!{sys.executable} -m pip install transformers==4.46.3

!python -c "import transformers; print(transformers.__version__)"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 76.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 96.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.10.1
    Uninstalling huggingface_hub-1.10.1:
      Successfully uninstalled huggingface_hub-1.10.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can

In [39]:
import sys

print(sys.executable)

/usr/bin/python3


In [40]:
!{sys.executable} -c "import transformers; print(transformers.__version__)"

4.46.3


In [43]:
code = r'''
from datasets import load_dataset
from transformers import pipeline

train = load_dataset("csv", data_files="/kaggle/input/datasets/tiwarikanaksuraj/train-csv/train (1).csv")["train"]

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small"
)

text = (
    f"Question: {train[0]['prompt']}. "
    f"Is the correct answer A: {train[0]['A']} "
    f"or B: {train[0]['B']}? "
    f"Answer with just the letter A or B."
)

result = generator(text, max_new_tokens=5)
print(result)
'''

import subprocess
import sys

subprocess.run([sys.executable, "-c", code])

2026-07-02 16:03:53.639509: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1783008233.664782     275 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783008233.672579     275 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783008233.692214     275 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783008233.692265     275 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783008233.692272     275 computation_placer.cc:177] computation placer alr

[{'generated_text': 'B'}]


CompletedProcess(args=['/usr/bin/python3', '-c', '\nfrom datasets import load_dataset\nfrom transformers import pipeline\n\ntrain = load_dataset("csv", data_files="/kaggle/input/datasets/tiwarikanaksuraj/train-csv/train (1).csv")["train"]\n\ngenerator = pipeline(\n    "text2text-generation",\n    model="google/flan-t5-small"\n)\n\ntext = (\n    f"Question: {train[0][\'prompt\']}. "\n    f"Is the correct answer A: {train[0][\'A\']} "\n    f"or B: {train[0][\'B\']}? "\n    f"Answer with just the letter A or B."\n)\n\nresult = generator(text, max_new_tokens=5)\nprint(result)\n'], returncode=0)